In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import quantus

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import gc
import pympler

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:
#A = np.zeros((8,135,2,60,900))
#size_in_gb = A.nbytes / 1e9
#print(f"Size of A: {size_in_gb:.2f} GB")

#import itertools
#S = list(itertools.combinations([1,2,13,24,27,29,34,42,43,46,52,56,57,60,62,67,69,72,73,80],2))

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/frequency_power_data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,29,34,42,43,52,60,62,67,69,80]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)      



    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def gradshap_explainer(
    model, inputs, targets, abs=False, normalise=False, *args, **kwargs
) -> np.array:
    """Wrapper aorund captum's GradShap implementation."""
    

    gc.collect()
    torch.cuda.empty_cache()

    # Set model in evaluate mode.
    model.to(kwargs.get("device", None))
    model.eval()

    #inputs = torch.from_numpy(inputs)
    #inputs = inputs.to(kwargs.get("device", None)).float()


    baselines = torch.zeros_like(inputs).to(kwargs.get("device", None)).float()
    gs = GradientShap(model)
    explanation = (
        gs
        .attribute(inputs=inputs, target=targets, baselines=baselines)
    ).cpu().data

    gc.collect()
    torch.cuda.empty_cache()

    if normalise:
        explanation = quantus.normalise_func.normalise_by_negative(explanation)

    if isinstance(explanation, torch.Tensor):
        if explanation.requires_grad:
            return explanation.cpu().detach().numpy()
        return explanation.cpu().numpy()

    return explanation

In [ ]:
def load_data_set(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index =  subject_index
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    print(all_epochs.shape)
    
    return all_epochs, labels_raw, ch_names

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

In [ ]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/finetune_model_weights"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_pretrain_subject_index_{subject_index}_start_idx_{start_index}.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model


to compare models of two subjects, take ~100 samples of any other subjects and generated explanations on those samples using the models of the 2 subjects

plan:

    - data set for subject comparison

    - load models of the two subjects at the same timestep

    - generate explanation with both models on the dataset

    - compute similarity of generated explanations


In [ ]:
def create_dataset(subject1=1,subject2=2, subsample=100):
    # create dataset where 10 trials are subsampled from all subjects except subject1 and subject2
    cfg = load_config()
    test_subjects = cfg.dataset.test_subject_indices
    combined_subjects = [subj for subj in test_subjects if subj not in [subject1, subject2]]
    all_data = np.zeros((len(combined_subjects), subsample, 60, 900))
    #print(f"all_data shape {all_data.shape}\n")
    rng = np.random.default_rng()
    for i,subj in enumerate(combined_subjects):
        epochs, _,_ = load_data_set(subj)
        # subsample from the epochs
        all_data[i] = rng.choice(epochs, subsample, replace=False)
    all_data = all_data.reshape(-1, 60, 900)

    return all_data

In [ ]:
def compute_explanations(model1, model2, dataset):
    # computes the explantion for the dataset given the two models
    # for the models finetuned models at different time steps are expected
    # len dataset x different explanations x channels x timepoints
    explanations = np.zeros((len(dataset), 2, 60, 900))
    batch_size = 50
    for i in range(0, dataset.shape[0], batch_size):
        batch_data = dataset[i:i+batch_size]
        batch_data = torch.from_numpy(batch_data).float().to(device)
        targets = torch.zeros(batch_data.shape[0]).long().to(device)
        explanation1 = gradshap_explainer(model1, batch_data, targets=targets)
        explanation2 = gradshap_explainer(model2, batch_data, targets=targets)
        explanations[i:i+batch_size,0] = explanation1
        explanations[i:i+batch_size,1] = explanation2

    # Handle the last batch if it's smaller than batch_size
    if dataset.shape[0] % batch_size != 0:
        batch_data = dataset[-(dataset.shape[0] % batch_size):]
        batch_data = torch.from_numpy(batch_data).float().to(device)
        targets = torch.zeros(batch_data.shape[0]).long().to(device)
        explanation1 = gradshap_explainer(model1, batch_data, targets=targets)
        explanation2 = gradshap_explainer(model2, batch_data, targets=targets)
        explanations[-(dataset.shape[0] % batch_size):,0] = explanation1
        explanations[-(dataset.shape[0] % batch_size):,1] = explanation2

    return explanations



In [ ]:
from scipy.stats import spearmanr
def compute_pairwise_rank_correlation(data, steps, subj1_index=0, subj2_index=1):
    #print(f"data shape {data.shape}")
    results = {}
    correlation_stats = np.zeros(len(steps))
    pvals = np.zeros((len(steps)))
    for time_step in range(data.shape[0]):
        #print(f"t {t}")
        subject_1_data = data[time_step,:,0].reshape(-1)
        subjcet_2_data = data[time_step,:,1].reshape(-1)
        # compute rank correlations subject1 and subject2
        correlation_stats[time_step], pvals[time_step] = spearmanr(subject_1_data, subjcet_2_data)
        
        #print(f"correlation {correlation} pval {pval}")
    results["correlations"] = correlation_stats
    results["pvals"] = pvals

    np.save(f"pairwise_rank_correlation_{subj1_index}_{subj2_index}.npy", results)

    #return correlation_stats, pvals


In [ ]:
from scipy.stats import spearmanr
def compute_pairwise_rank_correlation_timestep(data, steps, subj1_index=0, subj2_index=1):
    #print(f"data shape {data.shape}")
    results = {}
    correlation_stats = np.zeros((len(steps), data.shape[1]))
    pvals = np.zeros((len(steps), data.shape[1]))
    for time_step in range(data.shape[0]):
        for trial in range(data.shape[1]):
            #print(f"t {t}")
            subject_1_data = data[time_step,trial,0].reshape(-1)
            subjcet_2_data = data[time_step,trial,1].reshape(-1)
            # compute rank correlations subject1 and subject2
            correlation_stats[time_step, trial], pvals[time_step, trial] = spearmanr(subject_1_data, subjcet_2_data)
        
            #print(f"correlation {correlation} pval {pval}")
    results["correlations"] = correlation_stats
    results["pvals"] = pvals

    np.save(f"pairwise_rank_correlation_{subj1_index}_{subj2_index}_trial_wise.npy", results)

    #return correlation_stats, pvals

In [ ]:
steps = np.arange(100, 450, 50)
steps = np.concatenate((steps, [449]))
len(steps)

In [ ]:
import itertools
def compute_explanations_all_timesteps():
    cfg = load_config()
    subject_pairs = list(itertools.combinations(cfg.dataset.test_subject_indices, 2))
    #steps = np.arange(100,451,50)
    steps = np.arange(100, 450, 50)
    steps = np.concatenate((steps, [449]))

    for pair in tqdm(subject_pairs):
        dataset = create_dataset(pair[0], pair[1], subsample=10)
        # all_explanations_for_pair shape 300 finetune_steps x size dataset x 2 models x 60 channels x 900 timepoints
        # only look at fine tuned models every 50 start indices (otherwise to much data to store)
        # same argument for only subsampling 10 trials from every subject not in the pair
        all_explanations_for_pair = np.zeros((len(steps), dataset.shape[0], 2, 60, 900))

        #print(f" all explanations shape {all_explanations_for_pair.shape}")
        
        for step, start_index in enumerate(steps):
            #print(f"start index {start_index}")
            model1 = load_model(cfg, start_index=start_index, subject_index=pair[0])
            model2 = load_model(cfg, start_index=start_index, subject_index=pair[1])
            explanations = compute_explanations(model1, model2, dataset)
            #if start_index == 
 
            all_explanations_for_pair[step] = explanations
        compute_pairwise_rank_correlation(all_explanations_for_pair, steps, subj1_index=pair[0], subj2_index=pair[1])


        gc.collect()
        torch.cuda.empty_cache()

       
    

In [ ]:
#compute_explanations_all_timesteps()

In [ ]:
#compute the pairwise rank correlation for all subject pairs
cfg = load_config()
subject_pairs = list(itertools.combinations(cfg.dataset.test_subject_indices, 2))
correlations = np.zeros((len(subject_pairs), 8))
pvals = np.zeros((len(subject_pairs), 8))
for i, pair in enumerate(subject_pairs):
    correlations[i], pvals[i] = compute_pairwise_rank_correlation(pair[0], pair[1])
    print(f"pair {pair} correlation {correlations[i]} pvals {pvals[i]}")
np.save("correlations.npy", correlations)


get separate correlations and pvals for each time-step and also plot the similarity for different time steps
Problem: So far only agreement of models in first 450 time steps has been measured. Also check agreement of model for subject x in time t whith model for subject y at time s

In [ ]:
cfg = load_config()

In [ ]:
a = np.load("pairwise_rank_correlation_1_2.npy", allow_pickle=True).item()

In [ ]:
a

In [ ]:
def get_rank_correlations_timestep(t=0):
    subject_pairs = list(itertools.combinations(cfg.dataset.test_subject_indices, 2))
    correlations = np.zeros(len(subject_pairs))
    pvals = np.zeros(len(subject_pairs))
    for i, pair in enumerate(subject_pairs):
        a = np.load(f"pairwise_rank_correlation_{pair[0]}_{pair[1]}.npy", allow_pickle=True).item()
        correlations[i] = a["correlations"][t]
        pvals[i] = a["pvals"][t]
    return correlations, pvals

In [ ]:
c,p = get_rank_correlations_timestep(0)

In [ ]:
c8,p8 = get_rank_correlations_timestep(7)